<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# TelcomCI — RH & Masse Salariale
## Notebook 1 — Contexte, Découverte des Données & Premiers KPIs
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Entreprise** | TelcomCI — opérateur télécom, Abidjan, Côte d'Ivoire |
| **Période** | Janvier 2021 — Juin 2024 |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL) |
| **Durée estimée** | 3h à 4h |

---
> TelcomCI est un opérateur télécom ivoirien avec **510 employés** répartis dans **10 départements**. La DRH fait face à trois problèmes urgents :
>
> *"Nous ne savons pas exactement combien nous coûte la masse salariale par département. Notre taux de turnover est trop élevé et nous ne savons pas pourquoi. Les absences non justifiées ont augmenté sans que personne ne l'ait détecté."*

---
## 0. Mise en place de l'environnement

In [1]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import os, sys, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/rh_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

📁 Environnement : Local
📁 Dossier       : ./outputs/
Configuration chargée ✅


---
## 1. Connexion DuckDB et chargement des tables

### MÉTHODE — Ordre de chargement (dépendances)

En SQL Server, les tables avec clés étrangères (FK) ne peuvent être créées qu'après les tables qu'elles référencent. Dans DuckDB, les FK ne sont pas enforced mais on conserve l'ordre logique pour la lisibilité :

```
1. departements   (aucune FK)
2. employes        (→ departements)
3. salaires        (→ employes, departements)
4. absences        (→ employes, departements)
5. recrutements    (→ departements)
6. evaluations     (→ employes, departements)
```

> **DuckDB :** `read_csv_auto()` remplace `BULK INSERT`. Il charge le CSV directement depuis GitHub, infère les types automatiquement et crée la table en une seule instruction — zéro configuration.

In [2]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/'

conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE departements  AS SELECT * FROM read_csv_auto('{BASE_URL}departements.csv');
    CREATE TABLE employes      AS SELECT * FROM read_csv_auto('{BASE_URL}employes.csv',      timestampformat='%Y-%m-%d');
    CREATE TABLE salaires      AS SELECT * FROM read_csv_auto('{BASE_URL}salaires.csv');
    CREATE TABLE absences      AS SELECT * FROM read_csv_auto('{BASE_URL}absences.csv',      timestampformat='%Y-%m-%d');
    CREATE TABLE recrutements  AS SELECT * FROM read_csv_auto('{BASE_URL}recrutements.csv',  timestampformat='%Y-%m-%d');
    CREATE TABLE evaluations   AS SELECT * FROM read_csv_auto('{BASE_URL}evaluations.csv');
""")

result = conn.execute("""
    SELECT 'departements' AS table_name, COUNT(*) AS nb_lignes, 10   AS attendu FROM departements UNION ALL
    SELECT 'employes',                   COUNT(*),              510              FROM employes     UNION ALL
    SELECT 'salaires',                   COUNT(*),              17319            FROM salaires     UNION ALL
    SELECT 'absences',                   COUNT(*),              622              FROM absences     UNION ALL
    SELECT 'recrutements',               COUNT(*),              350              FROM recrutements UNION ALL
    SELECT 'evaluations',                COUNT(*),              1221             FROM evaluations
""").df()
display(result)
print('✅ 6 tables chargées')

,table_name,nb_lignes,attendu
0,departements,10,10
1,employes,510,510
2,salaires,17319,17319
3,absences,622,622
4,recrutements,350,350
5,evaluations,1221,1221


✅ 6 tables chargées


In [4]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## 2. Exploration des tables

### MÉTHODE — Comprendre le schéma de relations

Avant d'analyser, il faut connaître les clés de jointure entre tables :

```
departements
    ↓ departement_id
employes ──────────────────────────┐
    ↓ employe_id                   ↓ departement_id
salaires    absences    evaluations    recrutements
```

> **DuckDB :** `LIMIT 5` remplace `TOP 5` de SQL Server. La syntaxe `LIMIT` s'écrit en fin de requête, après `ORDER BY`.

In [5]:
%%sql
-- Explorer les 5 premières lignes de chaque table
SELECT * FROM departements LIMIT 5

In [6]:
%%sql
-- Schéma complet des colonnes (équivalent INFORMATION_SCHEMA)
SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## 3. Diagnostic qualité

### MÉTHODE — Pourquoi diagnostiquer avant d'analyser ?

Analyser des données sans vérifier leur qualité donne des KPIs faux. Exemple : un salaire négatif inclus dans une moyenne salariale la fausse sans qu'aucune erreur SQL ne soit levée. La règle : **toujours diagnostiquer avant d'analyser.**

Le pattern `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` compte les lignes qui satisfont une condition sans filtrer les autres — on voit ainsi les anomalies en proportion du total.

In [23]:
%%sql 
-- DIAGNOSTIC 1 : Valeurs NULL et anomalies dans employes
-- TODO : compter total_employes, null_email, null_salaire, null_dept
-- TODO : compter salaires_aberrants (<= 0), salaires_negatifs (< 0), salaires_nuls (= 0)
-- Indice : SUM(CASE WHEN condition THEN 1 ELSE 0 END)
SELECT
    COUNT(*)  AS total_employes
    -- TODO
FROM employes

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [24]:
%%sql 
-- DIAGNOSTIC 2 : Emails dupliqués
-- TODO : trouver les emails qui apparaissent plus d'une fois
-- Indice : GROUP BY email + HAVING COUNT(*) > 1
SELECT
    email,
    COUNT(*) AS nb_occurrences
FROM employes
WHERE email IS NOT NULL
-- TODO : ajouter GROUP BY, HAVING et ORDER BY

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
%%sql 
-- DIAGNOSTIC 3 : Salaires nets négatifs dans la table salaires
-- TODO : compter total_lignes, nets_negatifs (salaire_net < 0), non_payes
SELECT
    COUNT(*) AS total_lignes
    -- TODO
FROM salaires

In [26]:
%%sql 
-- Détail des salaires nets négatifs
-- TODO : afficher salaire_id, nom de l'employé, periode, brut, prime, cotisations, impot, net
-- Indice : JOIN employes sur employe_id | WHERE salaire_net < 0
-- Indice DuckDB : prenom || ' ' || nom pour concaténer
SELECT
    s.salaire_id
    -- TODO
FROM salaires s
JOIN employes e ON s.employe_id = e.employe_id
WHERE -- TODO

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [27]:
%%sql 
-- DIAGNOSTIC 4 : Absences avec nb_jours négatif
-- TODO : compter total_absences, jours_aberrants (<= 0), jours_negatifs (< 0)
-- TODO : calculer min_jours, max_jours, moy_jours
SELECT
    COUNT(*) AS total_absences
    -- TODO
FROM absences

In [28]:
%%sql
-- Vérification cohérence des dates
SELECT COUNT(*) AS incoherences_dates
FROM absences
WHERE date_fin < date_debut

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [29]:
%%sql 
-- DIAGNOSTIC 5 : Résumé synthétique toutes tables
SELECT 'employes'      AS table_name, 510   AS attendu, COUNT(*) AS reel FROM employes   UNION ALL
SELECT 'salaires',                    17319,             COUNT(*) FROM salaires    UNION ALL
SELECT 'absences',                    622,               COUNT(*) FROM absences    UNION ALL
SELECT 'recrutements',                350,               COUNT(*) FROM recrutements UNION ALL
SELECT 'evaluations',                 1221,              COUNT(*) FROM evaluations

---
## 4. Premiers KPIs RH

### KPI 1 — Effectif actif par département

### MÉTHODE
`LEFT JOIN` entre `departements` et `employes` avec filtre `statut = 'Actif'` permet d'inclure les départements même si leur effectif actif est zéro. `ROUND(effectif_actif * 100.0 / SUM(...) OVER(), 1)` calcule le pourcentage avec une window function — une seule requête suffit, pas besoin de sous-requête.

In [30]:
%%sql 
-- KPI 1 : Effectif actif par département avec écart vs cible
-- TODO : compter effectif_actif (statut = 'Actif') par département
-- TODO : calculer ecart_cible = effectif_actif - effectif_cible
-- TODO : calculer pct_total avec SUM(...) OVER() window function
-- Indice : LEFT JOIN pour inclure les départements à effectif 0
SELECT
    d.departement_id,
    d.nom            AS departement,
    d.effectif_cible
    -- TODO
FROM departements d
LEFT JOIN employes e
    ON  d.departement_id = e.departement_id
    AND e.statut = 'Actif'
GROUP BY d.departement_id, d.nom, d.effectif_cible
ORDER BY effectif_actif DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
### KPI 2 — Masse salariale (juin 2024)

### MÉTHODE
`SUM(salaire_brut) + SUM(prime)` = masse salariale brute totale. Ne pas confondre avec `SUM(salaire_brut + prime)` qui donne le même résultat mais peut poser des problèmes si certaines primes sont NULL — SQL traite NULL comme 0 dans `SUM()` mais pas dans une addition arithmétique.

In [32]:
%%sql 
-- KPI 2 : Masse salariale juin 2024 par département
-- TODO : compter nb_employes_payes, masse_brute, total_primes, masse_totale_fcfa, salaire_brut_moyen
-- Indice : filtrer annee = 2024 AND mois = 6 AND statut_paiement = 'Payé'
-- Indice : masse_totale = SUM(salaire_brut) + SUM(prime)
SELECT
    d.nom AS departement
    -- TODO
FROM salaires s
JOIN departements d ON s.departement_id = d.departement_id
WHERE -- TODO
GROUP BY d.departement_id, d.nom
ORDER BY masse_totale_fcfa DESC

In [35]:
%%sql 
-- KPI 2 global : agrégats pour juin 2024
-- TODO : employes_payes, masse_brute_totale, primes_totales, masse_totale_fcfa,
--        masse_nette_totale, salaire_brut_moyen
SELECT
    -- TODO
FROM salaires
WHERE annee = 2024
  AND mois  = 6
  AND statut_paiement = 'Payé'

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
### KPI 3 — Taux de turnover par département

### MÉTHODE
**Taux de turnover** = (Nb départs / Effectif total) × 100. C'est un indicateur de risque RH : un taux > 15% signale un département en tension. `RANK() OVER (ORDER BY taux DESC)` classe les départements du plus risqué au moins risqué sans réduire le nombre de lignes.

In [36]:
%%sql
-- KPI 3 : Taux de turnover par département avec RANK()
-- TODO : compter effectif_total, nb_departs (statut = 'Inactif')
-- TODO : calculer taux_turnover_pct = nb_departs * 100.0 / NULLIF(effectif_total, 0)
-- TODO : ajouter RANK() OVER (ORDER BY taux DESC) AS rang_risque
SELECT
    d.nom AS departement
    -- TODO
FROM departements d
JOIN employes e ON d.departement_id = e.departement_id
GROUP BY d.departement_id, d.nom
ORDER BY rang_risque

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
### KPI 4 — Répartition genre et profil démographique

In [37]:
%%sql 
-- KPI 4 : Répartition genre par département (actifs uniquement)
-- TODO : compter hommes (genre='M'), femmes (genre='F'), pct_femmes
-- Indice : SUM(CASE WHEN genre='F' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0)
SELECT
    d.nom AS departement,
    COUNT(e.employe_id) AS effectif
    -- TODO
FROM departements d
JOIN employes e ON d.departement_id = e.departement_id
WHERE e.statut = 'Actif'
GROUP BY d.departement_id, d.nom
ORDER BY pct_femmes DESC

In [38]:
%%sql 
-- Types de contrat avec pourcentage
-- TODO : compter nb_employes par type_contrat + pct avec SUM() OVER()
SELECT
    type_contrat
    -- TODO
FROM employes
GROUP BY type_contrat
ORDER BY nb_employes DESC

In [39]:
%%sql 
-- Âge moyen et ancienneté des actifs
-- TODO : calculer age_moyen et anciennete_moy_ans
-- Indice DuckDB : DATE_DIFF('year', date_naissance, CURRENT_DATE)
-- Indice DuckDB : DATE_DIFF('day', date_embauche, CURRENT_DATE) / 365.0
SELECT
    -- TODO
FROM employes
WHERE statut = 'Actif'

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
### KPI 5 — Absences injustifiées par département

### MÉTHODE
Le champ `justifiee` est un **booléen** (0/1). Filtrer `WHERE justifiee = false` retourne les absences non justifiées.

> **DuckDB :** selon la façon dont le CSV est lu, `justifiee` peut être de type `BOOLEAN` ou `INTEGER`. Les deux filtres `justifiee = false` et `justifiee = 0` sont équivalents.

In [40]:
%%sql 
-- KPI 5 : Absences injustifiées par département
-- TODO : compter nb_absences_injust, jours_injustifies (justifiee = false)
-- TODO : calculer pct_total sur le total de jours injustifiés (sous-requête)
SELECT
    d.nom AS departement
    -- TODO
FROM absences a
JOIN departements d ON a.departement_id = d.departement_id
WHERE a.justifiee = false
  AND a.nb_jours  > 0
GROUP BY d.departement_id, d.nom
ORDER BY jours_injustifies DESC

In [41]:
%%sql 
-- Comparaison justifiées vs injustifiées
-- TODO : nb_absences, total_jours, pct par type
-- Indice : CASE WHEN justifiee = true THEN 'Justifiée' ELSE 'Non justifiée' END
SELECT
    -- TODO
FROM absences
WHERE nb_jours > 0
GROUP BY justifiee

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Bilan du Notebook 1

### Anomalies identifiées

| Anomalie | Nb | Table |
|---|---|---|
| Salaires base aberrants (négatif ou nul) | **2** | employes |
| Emails dupliqués | **2** | employes |
| Salaires nets négatifs | **3** | salaires |
| Paiements 'En attente' | **179** | salaires |
| Absences nb_jours négatifs | **3** | absences |

### KPIs de référence

| KPI | Valeur |
|---|---|
| Effectif actif total | **450** |
| Masse salariale brute (juin 2024) | **457 689 542 FCFA** |
| Salaire brut moyen | **1 030 832 FCFA** |
| Taux de turnover global | **11.8%** |
| Département turnover le plus élevé | **Marketing (23.3%)** |
| Répartition M/F | **64.7% / 35.3%** |
| Dept absences injustifiées N°1 | **Technique & Réseau (106j)** |

### Adaptations DuckDB vs SQL Server

| SQL Server | DuckDB |
|---|---|
| `TOP N` | `LIMIT N` |
| `BULK INSERT` | `read_csv_auto()` |
| `DATEDIFF(YEAR, d, GETDATE())` | `DATE_DIFF('year', d, CURRENT_DATE)` |
| `prenom + ' ' + nom` | `prenom \|\| ' ' \|\| nom` |
| `BIT` (0/1) | `BOOLEAN` (true/false) |
| `PRINT 'message'` | `print('message')` Python |

**Pour le Notebook 2 :** créer les 3 vues de nettoyage, analyser la masse salariale mensuelle avec LAG(), calculer le taux de turnover avec CTEs et window functions, construire la matrice performance × salaire avec NTILE(4).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.